In [1]:
import pandas as pd

In [4]:
df = pd.read_parquet("wildchat_dedup.parquet")
print(f"Loaded {len(df):,} rows, {df['state'].nunique():,} unique states")

# ── Compute per-state stats ───────────────────────────────────────────────────
state_stats = df.groupby("state").agg(
    row_count=("user_text", "count"),
    unique_ips=("hashed_ip", "nunique"),
).reset_index()

# ── Filter: keep states meeting BOTH thresholds ───────────────────────────────
MIN_ROWS = 100
MIN_IPS  = 40

valid_states = state_stats.loc[
    (state_stats["row_count"] >= MIN_ROWS) &
    (state_stats["unique_ips"] >= MIN_IPS),
    "state",
]

print(f"\nState filter (min rows={MIN_ROWS}, min unique IPs={MIN_IPS}):")
print(f"  States before : {len(state_stats):,}")
print(f"  States kept   : {len(valid_states):,}")
print(f"  States dropped: {len(state_stats) - len(valid_states):,}")

# ── Apply filter ──────────────────────────────────────────────────────────────
df_filtered = df[df["state"].isin(valid_states)].reset_index(drop=True)

print(f"\nRows before : {len(df):,}")
print(f"Rows kept   : {len(df_filtered):,}")
print(f"Rows dropped: {len(df) - len(df_filtered):,}")

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = "wildchat_state_filtered.parquet"
df_filtered.to_parquet(out_path, index=False)
print(f"\nSaved → {out_path}")

Loaded 69,842 rows, 51 unique states

State filter (min rows=100, min unique IPs=40):
  States before : 51
  States kept   : 36
  States dropped: 15

Rows before : 69,842
Rows kept   : 68,668
Rows dropped: 1,174

Saved → wildchat_state_filtered.parquet


In [2]:
import pandas as pd
from transformers import BertTokenizer

# Load data
df = pd.read_parquet("wildchat_state_filtered.parquet")

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Count tokens for each row
def count_tokens(text):
    if pd.isna(text):
        return 0
    tokens = tokenizer.encode(text, add_special_tokens=True, truncation=False)
    return len(tokens)

print(f"Total rows: {len(df)}")
print("Counting tokens per row (this may take a moment)...")

df["token_count"] = df["user_text"].apply(count_tokens)

# Check how many rows exceed 512 tokens
over_512 = (df["token_count"] > 512).sum()
total = len(df)

print(f"\nRows with > 512 tokens: {over_512:,} ({over_512/total*100:.2f}%)")
print(f"Rows with <= 512 tokens: {total - over_512:,} ({(total - over_512)/total*100:.2f}%)")
print(f"\nToken count statistics:")
print(df["token_count"].describe(percentiles=[.25, .5, .75, .90, .95, .99]))

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Token indices sequence length is longer than the specified maximum sequence length for this model (735 > 512). Running this sequence through the model will result in indexing errors


Total rows: 68668
Counting tokens per row (this may take a moment)...

Rows with > 512 tokens: 9,316 (13.57%)
Rows with <= 512 tokens: 59,352 (86.43%)

Token count statistics:
count    68668.000000
mean       251.807305
std       1104.575156
min          3.000000
25%         16.000000
50%         39.000000
75%        160.000000
90%        716.000000
95%        955.000000
99%       2749.990000
max      83748.000000
Name: token_count, dtype: float64


In [3]:
# ── Truncate long texts: first 250 + last 250 tokens ─────────────────────────
# For texts exceeding 512 BERT tokens, we combine the first 250 and last 250
# content tokens (excluding [CLS]/[SEP] special tokens) and decode back to
# a string. Texts within the 512-token limit are left unchanged.

HEAD_TOKENS = 250
TAIL_TOKENS = 250

def truncate_to_head_tail(text, head=HEAD_TOKENS, tail=TAIL_TOKENS):
    """Return text unchanged if <= 512 tokens; otherwise decode first `head`
    and last `tail` content tokens (no special tokens) back to a string."""
    if pd.isna(text):
        return text

    # Encode without special tokens so indexing is purely over content tokens
    token_ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)

    # +2 accounts for [CLS] and [SEP] that BERT prepends/appends
    if len(token_ids) + 2 <= 512:
        return text

    combined_ids = token_ids[:head] + token_ids[-tail:]
    return tokenizer.decode(combined_ids, skip_special_tokens=True,
                            clean_up_tokenization_spaces=True)


print("Applying head-tail truncation to rows with > 512 tokens...")
df["user_text"] = df["user_text"].apply(truncate_to_head_tail)

# ── Save ──────────────────────────────────────────────────────────────────────
KEEP_COLS = ["conversation_hash", "model", "timestamp", "hashed_ip", "user_text", "state"]

out_path = "wildchat_bertopic.parquet"
df[KEEP_COLS].to_parquet(out_path, index=False)
print(f"\nSaved → {out_path}")
print(f"Columns in output: {KEEP_COLS}")

Applying head-tail truncation to rows with > 512 tokens...

Saved → wildchat_bertopic.parquet
